# Tutorial - AIME 2026: From LLMs to DNA
## 🦠 Notebook 3. SARS-CoV-2 Variant Prioritization Using Genomic Language Models

This notebook demonstrates an end-to-end bioinformatics pipeline utilizing the specialized **Masked Language Model (MLM)**—MiniBERT previously adapted on SARS-CoV-2 genomes prior to the Omicron era. The main goal is to align unseen Omicron against the standard ancestral reference genome (Wuhan Reference), isolate emerging point mutations, analyze their impact at the protein level (Synonymous vs. Non-synonymous amino acid substitutions), and prioritize them using the model's **$\Delta$ Log-Likelihood Surprise Score**.

## 1.Install dependencies

In [ ]:
!pip install biopython pandas torch transformers

import os
import urllib.request
import torch
import pandas as pd
import numpy as np
from Bio import SeqIO
from Bio import Align
from transformers import BertConfig, BertForMaskedLM, PreTrainedTokenizerFast

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2.Load Genomic Coordinates and Reference Sequence

Here, we read the ancestral **Wuhan-Hu-1 reference sequence** and normalize the base alphabet to uppercase.

Additionally, we define a standard index mapping (`GENOMIC_REGIONS`) representing the absolute, **1-based standard genomic coordinates** for every major Open Reading Frame (ORF) and functional gene structure across the ~29,903 bp SARS-CoV-2 genome. The utility function `get_genomic_region()` takes an active mutation coordinate and automatically categorizes it into its respective gene or notes it as intergenic/non-coding text.

In [ ]:
# Load Wuhan reference sequence from GitHub
reference_url = "https://raw.githubusercontent.com/pabloarozarena/aime2026-T7-genomic-llms/main/data/sarscov2_reference.fasta"
reference_fasta = "sarscov2_reference.fasta"

urllib.request.urlretrieve(reference_url, reference_fasta)

wuhan_record = SeqIO.read(reference_fasta, "fasta")
wuhan_seq = str(wuhan_record.seq).upper()

print(f"Wuhan Reference Loaded from GitHub. Length: {len(wuhan_seq)} bp")

# SARS-CoV-2 standard 1-based genomic coordinates
GENOMIC_REGIONS = {
    "ORF1ab": (266, 21555),
    "Spike (S)": (21563, 25384),
    "ORF3a": (25393, 26220),
    "Envelope (E)": (26245, 26472),
    "Membrane (M)": (26523, 27191),
    "ORF6": (27202, 27387),
    "ORF7a": (27394, 27759),
    "ORF7b": (27756, 27887),
    "ORF8": (27894, 28259),
    "Nucleocapsid (N)": (28274, 29533),
    "ORF10": (29558, 29674)
}

def get_genomic_region(position):
    """Maps a 1-based genomic coordinate to its corresponding gene structure."""
    for region, (start, end) in GENOMIC_REGIONS.items():
        if start <= position <= end:
            return region
    return "Non-coding / Intergenic"


## 3.Instantiate Tokenizer and Reconstruct Model Architecture

In this cell, the user selects either the **character** or **kmer** SARS-CoV-2 MiniBERT model. The tokenizer files and checkpoint weights are downloaded directly from the GitHub repository, then the MiniBERT architecture is reconstructed and loaded in evaluation mode.


In [ ]:
# Choose which SARS-CoV-2 MiniBERT model to use: "character" or "kmer"
MODEL_CHOICE = "character"

github_base = "https://raw.githubusercontent.com/pabloarozarena/aime2026-T7-genomic-llms/main/models/sarscov2"

model_files = {
    "character": {
        "tokenizer_file": "char_tokenizer.json",
        "tokenizer_config_file": "char_tokenizer_config.json",
        "weights_file": "char_ckpt_epoch_10.pt"
    },
    "kmer": {
        "tokenizer_file": "kmer_tokenizer.json",
        "tokenizer_config_file": "kmer_tokenizer_config.json",
        "weights_file": "kmer_ckpt_epoch_10.pt"
    }
}

selected = model_files[MODEL_CHOICE]
local_dir = f"sarscov2_{MODEL_CHOICE}"
os.makedirs(local_dir, exist_ok=True)

tokenizer_path = os.path.join(local_dir, selected["tokenizer_file"])
tokenizer_config_path = os.path.join(local_dir, selected["tokenizer_config_file"])
weights_path = os.path.join(local_dir, selected["weights_file"])

urllib.request.urlretrieve(
    f"{github_base}/{MODEL_CHOICE}/tokenizer/{selected['tokenizer_file']}",
    tokenizer_path
)
urllib.request.urlretrieve(
    f"{github_base}/{MODEL_CHOICE}/tokenizer/{selected['tokenizer_config_file']}",
    tokenizer_config_path
)
urllib.request.urlretrieve(
    f"{github_base}/{MODEL_CHOICE}/weights/{selected['weights_file']}",
    weights_path
)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=tokenizer_path,
    tokenizer_config_file=tokenizer_config_path
)

if tokenizer.mask_token is None:
    tokenizer.add_special_tokens({
        "mask_token": "[MASK]",
        "pad_token": "[PAD]",
        "cls_token": "[CLS]",
        "sep_token": "[SEP]",
        "unk_token": "[UNK]"
    })

if MODEL_CHOICE == "kmer":
    dna_tokens = [
        tok for tok in tokenizer.get_vocab()
        if len(tok) > 1 and set(tok).issubset(set("ACGTN"))
    ]
    KMER_SIZE = min(set(len(tok) for tok in dna_tokens)) if dna_tokens else 6
    KMER_STRIDE = 1

def format_sequence(seq):
    seq = seq.upper()

    if MODEL_CHOICE == "kmer":
        return " ".join(
            seq[i:i + KMER_SIZE]
            for i in range(0, len(seq) - KMER_SIZE + 1, KMER_STRIDE)
        )

    return " ".join(list(seq))

config = BertConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=128,
    num_hidden_layers=2,
    num_attention_heads=2,
    intermediate_size=512,
    max_position_embeddings=512
)

model = BertForMaskedLM(config)

checkpoint = torch.load(weights_path, map_location=device)

if "state_dict" in checkpoint:
    model.load_state_dict(checkpoint["state_dict"])
elif "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()

print(f"Loaded SARS-CoV-2 {MODEL_CHOICE} tokenizer and model from GitHub.")


## 4.Load and Prepare Variant Cohorts

Here, we ingest our Omicron dataset. Real-world genetic sequencing reads often contain data artifacts labeled as `N` or `n`, which indicate an uncalled or ambiguous base readout from the sequencing instrument. Since language models can react unpredictably to excessive masking artifacts, we filter out all row strings containing `N` to maintain a robust, fully resolved dataset.

Once cleaned, we draw a deterministic sample sequence (`random_state=42`) representing an isolation line of the **Omicron Lineage** to prepare for downstream comparison.

In [ ]:
# Load SARS-CoV-2 dataset from GitHub
sarscov2_url = "https://raw.githubusercontent.com/pabloarozarena/aime2026-T7-genomic-llms/main/data/sarscov2.csv.gz"

df_test = pd.read_csv(sarscov2_url, compression="gzip")

# Keep only Omicron sequences
df_omicron = df_test[df_test["Variant"] == "Omicron"].copy()

# Remove sequences containing ambiguous bases
df_omicron_clean = df_omicron[
    ~df_omicron["sequence"].str.contains("N|n", na=True, regex=True)
].copy()

# Randomly select 50 clean Omicron sequences
df_omicron_sample = df_omicron_clean.sample(
    n=min(50, len(df_omicron_clean)),
    random_state=42
).reset_index(drop=True)

omicron_sequences = df_omicron_sample["sequence"].str.upper().tolist()

print(f"Total sequences: {len(df_test)}")
print(f"Omicron sequences: {len(df_omicron)}")
print(f"Clean Omicron sequences without Ns: {len(df_omicron_clean)}")
print(f"Selected Omicron sequences for Spike analysis: {len(omicron_sequences)}")
print(f"First selected sequence length: {len(omicron_sequences[0])} bp")

## 5.Sequence Alignment, Translation Logic, & Functional Context Scoring

This is the core analytical block of our engine. It performs three critical operations:
1. **Pairwise Global Alignment**: Aligns the full Omicron genome against the Wuhan string to accurately compute 1-to-1 matching position indices while accounting for indels or gaps.
2. **Dynamic Reading Frame Codon Reconstruction**: The function `get_amino_acid_change()` extracts the specific 3-nucleotide triplet context from the reference gene, inserts the Omicron single point mutation base into its correct relative offset position, translates both frames using the standard genetic table, and labels mutations as **Synonymous (Silent)** or **Non-Synonymous (Amino acid changing)**.
3. **Masked Language Modeling (MLM) Surprise Evaluation**: For every point mutation discovered, the system slices a local 500 bp structural context window surrounding the locus on the Omicron string. It masks the target mutation site (`[MASK]`), feeds it to MiniBERT, and extracts the raw output logits.

### Mathematical Definition of Model "Surprise" Score
We apply a Log-Softmax function over the model's logits to establish true probability vectors ($P$) for all four potential bases. The final score represents the shift in expected biological grammar:

$$\Delta \text{Log-Likelihood} = \log P(\text{Omicron Token} \mid \text{Context}) - \log P(\text{Wuhan Token} \mid \text{Context})$$

* **Highly Negative Delta ($\Delta \ll 0$)**: Implies the model strongly expected the historical Wuhan base and is heavily "surprised" by the new mutation. This points to positions under intense evolutionary pressure (e.g., driver mutations or antibody escape sites).

In [ ]:
from Bio import Align
from Bio.Seq import Seq
from tqdm.auto import tqdm

# Spike region in Wuhan reference
SPIKE_START = 21563
SPIKE_END = 25384

# Use a small margin to account for small shifts/deletions
SPIKE_MARGIN = 300

wuhan_spike_seq = wuhan_seq[SPIKE_START - 1:SPIKE_END]

def get_amino_acid_change(wuhan_full_seq, wuhan_pos_1based, region, mut_base):
    if region == "Non-coding / Intergenic":
        return "Non-coding", "-"

    start, end = GENOMIC_REGIONS[region]

    if wuhan_pos_1based < start or wuhan_pos_1based > end:
        return "Unknown", "-"

    wuhan_idx_0based = wuhan_pos_1based - 1
    orf_offset = wuhan_idx_0based - (start - 1)

    codon_start_offset = (orf_offset // 3) * 3
    codon_wuhan_start = (start - 1) + codon_start_offset
    codon_wuhan_end = codon_wuhan_start + 3

    if codon_wuhan_end > len(wuhan_full_seq):
        return "Unknown", "-"

    codon_w = wuhan_full_seq[codon_wuhan_start:codon_wuhan_end]

    if len(codon_w) != 3:
        return "Unknown", "-"

    mutation_codon_pos = orf_offset % 3

    codon_o_list = list(codon_w)
    codon_o_list[mutation_codon_pos] = mut_base
    codon_o = "".join(codon_o_list)

    try:
        aa_w = str(Seq(codon_w).translate())
        aa_o = str(Seq(codon_o).translate())
    except:
        return "Unknown", "-"

    aa_pos = (codon_start_offset // 3) + 1

    if aa_w == aa_o:
        return "Synonymous", f"{aa_w}{aa_pos}{aa_o} (Silent)"
    else:
        return "Non-Synonymous", f"{aa_w}{aa_pos}{aa_o}"


def build_masked_context(local_seq, target_local_idx, ref_base, mut_base):
    if MODEL_CHOICE == "character":
        tokens = list(local_seq)
        tokens[target_local_idx] = tokenizer.mask_token
        return " ".join(tokens), ref_base, mut_base

    token_starts = list(range(0, len(local_seq) - KMER_SIZE + 1, KMER_STRIDE))

    overlapping = [
        i for i, start in enumerate(token_starts)
        if start <= target_local_idx < start + KMER_SIZE
    ]

    if len(overlapping) == 0:
        return None, None, None

    target_token_idx = overlapping[len(overlapping) // 2]
    kmer_start = token_starts[target_token_idx]

    kmer_tokens = [
        local_seq[i:i + KMER_SIZE]
        for i in token_starts
    ]

    mut_token = kmer_tokens[target_token_idx]

    ref_token_list = list(mut_token)
    ref_token_list[target_local_idx - kmer_start] = ref_base
    ref_token = "".join(ref_token_list)

    kmer_tokens[target_token_idx] = tokenizer.mask_token

    return " ".join(kmer_tokens), ref_token, mut_token


aligner = Align.PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 1
aligner.mismatch_score = -1
aligner.open_gap_score = -3
aligner.extend_gap_score = -1

mutation_data = []
window_half_size = 250

model.eval()

print("Scanning Spike region in 50 random Omicron sequences...")

for seq_id, omicron_seq in enumerate(tqdm(omicron_sequences, desc="Analyzing Spike")):

    omicron_spike_seq = omicron_seq[
        max(0, SPIKE_START - 1 - SPIKE_MARGIN):
        min(len(omicron_seq), SPIKE_END + SPIKE_MARGIN)
    ]

    alignments = aligner.align(wuhan_spike_seq, omicron_spike_seq)
    best_alignment = alignments[0]

    wuhan_aligned, omicron_aligned = best_alignment

    wuhan_spike_idx = 0
    omicron_spike_idx = 0

    for char_w, char_o in zip(wuhan_aligned, omicron_aligned):

        if char_w != "-":
            wuhan_spike_idx += 1

        if char_o != "-":
            omicron_spike_idx += 1

        if char_w != "-" and char_o != "-" and char_w != char_o:

            if char_w in "ACGT" and char_o in "ACGT":

                wuhan_global_pos = SPIKE_START + wuhan_spike_idx - 1

                start_pos = max(0, omicron_spike_idx - 1 - window_half_size)
                end_pos = min(
                    len(omicron_spike_seq),
                    omicron_spike_idx - 1 + window_half_size + 1
                )

                local_context = omicron_spike_seq[start_pos:end_pos]
                target_local_idx = (omicron_spike_idx - 1) - start_pos

                masked_input_str, ref_token, mut_token = build_masked_context(
                    local_context,
                    target_local_idx,
                    char_w,
                    char_o
                )

                if masked_input_str is None:
                    continue

                inputs = tokenizer(masked_input_str, return_tensors="pt")

                mask_positions = (
                    inputs["input_ids"][0] == tokenizer.mask_token_id
                ).nonzero(as_tuple=True)[0]

                if len(mask_positions) == 0:
                    continue

                token_position = mask_positions[0].item()

                ref_token_id = tokenizer.convert_tokens_to_ids(ref_token)
                mut_token_id = tokenizer.convert_tokens_to_ids(mut_token)

                if ref_token_id == tokenizer.unk_token_id or mut_token_id == tokenizer.unk_token_id:
                    continue

                inputs = {k: v.to(device) for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = model(**inputs)
                    logits = outputs.logits[0, token_position]
                    log_probs = torch.log_softmax(logits, dim=-1)

                delta_score = (
                    log_probs[mut_token_id].item()
                    - log_probs[ref_token_id].item()
                )

                region = get_genomic_region(wuhan_global_pos)

                mut_type, aa_change = get_amino_acid_change(
                    wuhan_seq,
                    wuhan_global_pos,
                    region,
                    char_o
                )

                mutation_data.append({
                    "Sequence_ID": seq_id,
                    "Wuhan_Pos": wuhan_global_pos,
                    "Region": region,
                    "Mutation": f"{char_w}{wuhan_global_pos}{char_o}",
                    "Type": mut_type,
                    "AA_Change": aa_change,
                    "Delta_Log_Likelihood": delta_score
                })

df_results = pd.DataFrame(mutation_data)

print(f"Analysis complete. Calculated effects for {len(df_results)} Spike mutation instances.")

if len(df_results) > 0:
    print(f"Unique Spike mutations found: {df_results['Mutation'].nunique()}")

## 6.Rank, Format, and Display Top 50 Driver Mutations

In this final phase, the mutations are sorted in ascending order by their `Delta_Log_Likelihood` score, bringing the most negative, high-surprise variants directly to the top of the dataset.

In [ ]:
# Sort by the most "surprising" mutations, most negative delta first
df_results = df_results.sort_values(by="Delta_Log_Likelihood", ascending=True)

# Save full functional annotation output
df_results.to_csv("omicron_spike_variant_prioritization.csv", index=False)

# Format visual data frame
df_display = df_results.copy()

df_display = df_display[
    [
        "Sequence_ID",
        "Wuhan_Pos",
        "Region",
        "Mutation",
        "Type",
        "AA_Change",
        "Delta_Log_Likelihood"
    ]
]

df_display.columns = [
    "Sequence ID",
    "Wuhan Position (1-based)",
    "Viral Gene/Region",
    "Nucleotide Mutation",
    "Mutation Effect Type",
    "Amino Acid Substitution",
    "Model Surprise Score (Δ Log Likelihood)"
]

print("\n===== TOP 50 SURPRISING OMICRON SPIKE MUTATIONS (ANNOTATED PROTEIN LEVEL) =====")

df_display.head(50).style \
    .format({"Model Surprise Score (Δ Log Likelihood)": "{:.4f}"}) \
    .set_properties(**{"text-align": "center"}, subset=[
        "Sequence ID",
        "Wuhan Position (1-based)",
        "Nucleotide Mutation",
        "Mutation Effect Type",
        "Amino Acid Substitution"
    ]) \
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#f8f9fa"),
                ("color", "#212529"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "10px"),
                ("border", "1px solid #dee2e6")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("padding", "8px"),
                ("border", "1px solid #dee2e6")
            ]
        },
        {
            "selector": "tr:hover",
            "props": [
                ("background-color", "#f1f3f5")
            ]
        }
    ]) \
    .hide(axis="index")

## 7.External Validation and Molecular Surveillance

Congratulations! You have successfully leveraged a structural genomic language model to prioritize variants by biological novelty and model "surprise."

### Validating Your Highlights in Modern Omicron Strains
To verify whether the top 50 high-surprise mutations identified by your MiniBERT model have stabilized, expanded, or mutated further in newly emerging lineages, you can perform external validation using curated clinical databases.

1. Navigate to the **[Stanford Coronavirus Antiviral & Resistance Database (CoVDB)](https://covdb.stanford.edu/variants/)**.
2. Search for active and contemporary **Omicron sub-lineages** (e.g., *BA.1, BA.5, XBB, JN.1, KP.3*).
3. Cross-reference your `Model Surprise Score` findings with their real-world mutation frequency maps and clinical phenotype evidence:
   * **Correlate with Immune Escape:** Many of the highly negative outlier positions flagged by your model may represent critical fitness checkpoints where the virus is undergoing structural evolution to dodge neutralizing antibodies.
   * **Distinguish Passenger vs. Driver Mutations:** Check if your highly negative mutations are consistently maintained across modern lineages (signifying true evolutionary drivers) or if they were transient adaptations.